# A take-home with a written defense

Data Scientist track, lesson DS15. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** deliver a timed take-home on the app-events subscriptions (who cancels at the first renewal), with a scope note, a leakage check, a baseline, an honest evaluation and a limitations line, then defend it in writing and score yourself with the rubric printed in the lesson.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
The cells before "Your work" build the file the hiring team sent. Work in the empty cells and the text cells under "Your work", with a timer. The reference answer starts at the last heading: open it only after you finish, and
compare it with your own numbers. It runs from the file alone, so it gives the same output whether or not you did your work first. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import os
import subprocess
import sys
import tempfile
import urllib.request

import numpy as np
import pandas as pd

try:
    import duckdb
except ImportError:  # Colab usually has DuckDB; install it when it does not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 2026       # the seed of the whole Data Scientist track

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
LOCAL = "../../datasets/app-events/v1/"      # the same files, when this notebook runs inside a clone of the repository


def fetch(name):
    """The path of one CSV file: downloaded from the tag, or the local copy."""
    try:
        path = os.path.join(tempfile.mkdtemp(), name + ".csv")
        urllib.request.urlretrieve(BASE + name + ".csv", path)
        return path
    except Exception:
        return LOCAL + name + ".csv"


con = duckdb.connect()                       # a new, empty database every time this cell runs
con.execute(f"CREATE TABLE users AS FROM read_csv('{fetch('users')}', types={{'signup_date': 'DATE'}})")
con.execute(f"CREATE TABLE events AS FROM read_csv('{fetch('events')}', types={{'event_ts': 'TIMESTAMP', 'received_at': 'TIMESTAMP'}})")
con.execute(f"CREATE TABLE subscriptions AS FROM read_csv('{fetch('subscriptions')}', types={{'started_on': 'DATE', 'ended_on': 'DATE', 'updated_at': 'TIMESTAMP'}})")

# Known answer from the dataset README: 21,841 people, 177,379 events and 3,840 subscription rows.
assert con.execute("SELECT (SELECT COUNT(*) FROM users), (SELECT COUNT(*) FROM events), (SELECT COUNT(*) FROM subscriptions)").fetchall()[0] == (21841, 177379, 3840)

## The file the hiring team sent

Run this cell as it is. It prints the shape of the file and three rows.

In [ ]:
# The file the hiring team sent: one row for each person's first Plus subscription, built from the app-events tables. Run this cell as it is.
TAKE_HOME_SQL = """WITH first_sub AS (          -- one row for each person: their first Plus subscription, as it was first saved
  SELECT user_id, subscription_id, plan, started_on
  FROM subscriptions
  QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY started_on, subscription_id, updated_at) = 1
),
latest AS (                 -- the latest saved version of each subscription
  SELECT subscription_id, plan AS plan_now, ended_on, cancel_reason
  FROM subscriptions
  QUALIFY ROW_NUMBER() OVER (PARTITION BY subscription_id ORDER BY updated_at DESC, ended_on NULLS LAST) = 1
),
clean AS (                  -- retry copies removed: keep the first copy of each event
  SELECT *
  FROM events
  QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id, event_ts, event_name, platform ORDER BY received_at, event_id) = 1
),
bought AS (                 -- monthly subscriptions that started by May 31, with the purchase moment and the day-14 cutoff
  SELECT f.user_id, f.subscription_id, f.started_on, p.event_ts AS bought_ts, p.event_ts + INTERVAL 14 DAY AS cutoff
  FROM first_sub AS f
  JOIN clean AS p ON p.user_id = f.user_id AND p.event_name = 'purchase' AND CAST(p.event_ts AS DATE) = f.started_on
  WHERE f.plan = 'plus_monthly' AND f.started_on <= DATE '2026-05-31'
)
SELECT b.user_id, b.started_on, u.channel, u.region, s.platform, u.plan_at_signup,
       DATEDIFF('day', s.event_ts, b.bought_ts) AS days_to_buy,
       COUNT(*) FILTER (WHERE e.event_name IN ('session_start', 'add_expense') AND e.event_ts <= b.bought_ts) AS activity_before,
       COUNT(*) FILTER (WHERE e.event_name = 'session_start' AND e.event_ts > b.bought_ts AND e.event_ts <= b.cutoff AND e.received_at <= b.cutoff) AS visits_14d,
       COUNT(*) FILTER (WHERE e.event_name = 'add_expense' AND e.event_ts > b.bought_ts AND e.event_ts <= b.cutoff AND e.received_at <= b.cutoff) AS expenses_14d,
       COUNT(*) FILTER (WHERE e.event_name = 'plus_feature_used' AND e.event_ts > b.bought_ts AND e.event_ts <= b.cutoff AND e.received_at <= b.cutoff) AS plus_uses_14d,
       COUNT(*) FILTER (WHERE e.event_name = 'plus_feature_used') AS plus_uses,
       COUNT(*) FILTER (WHERE e.event_name IN ('session_start', 'add_expense')) AS active_events,
       l.plan_now, l.cancel_reason,
       CAST(l.ended_on IS NOT NULL AND l.ended_on <= b.started_on + 30 AS INTEGER) AS canceled
FROM bought AS b
JOIN users AS u ON u.user_id = b.user_id
JOIN clean AS s ON s.user_id = b.user_id AND s.event_name = 'signup_complete'
JOIN latest AS l ON l.subscription_id = b.subscription_id
LEFT JOIN clean AS e ON e.user_id = b.user_id
GROUP BY ALL
ORDER BY b.user_id"""
take_home = con.sql(TAKE_HOME_SQL).df()
print(take_home.shape, int(take_home["canceled"].sum()))
print(take_home.iloc[[0, 3, 4]].fillna("").to_string(index=False))

## Two helpers

In [ ]:
# Two helpers from earlier lessons. Run this cell as it is.
def recipe(cat, num):
    """The model of lesson DS2: one-hot text columns, scaled number columns, then a logistic regression."""
    parts = []
    if cat:
        parts.append(("cat", OneHotEncoder(handle_unknown="ignore"), cat))
    if num:
        parts.append(("num", StandardScaler(), num))
    return make_pipeline(ColumnTransformer(parts), LogisticRegression(max_iter=1000))


def auc_gain(p_new, p_base, y_true, resamples=2000):
    """ROC AUC of one score minus another on the same people, with a 95 percent bootstrap interval; both are scored on the same resamples."""
    rng = np.random.default_rng(SEED)
    y_true, a, b = np.asarray(y_true), np.asarray(p_new), np.asarray(p_base)
    gains = []
    for _ in range(resamples):
        i = rng.integers(0, len(y_true), len(y_true))
        gains.append(roc_auc_score(y_true[i], a[i]) - roc_auc_score(y_true[i], b[i]))
    low, high = np.percentile(gains, [2.5, 97.5])
    return [round(float(roc_auc_score(y_true, a) - roc_auc_score(y_true, b)), 3) + 0.0, round(float(low), 3) + 0.0, round(float(high), 3) + 0.0]

## Your work

Start a timer for 90 minutes. Write the scope note first, before you look at the file again.

### Your scope note

Double-click here and write it: the question in one sentence, the minutes for each step, two things you will not do, one assumption.

### Your columns, baseline and model

In [ ]:
# your work here

In [ ]:
# more working cells, as many as you need

### Your written defense

Double-click here and write it: the answer first, then the three questions of a reviewer (is the number real, could it be known in time, what do we do with it), and one limitations line.

## Reference answer: open only after you finish

The cells below read only the file above, so they give the same output whatever your own cells did.

### The frame

In [ ]:
# The frame: who is in the file and who is left out, counted from the subscriptions table.
counts = con.execute("""WITH first_sub AS (
  SELECT plan, started_on FROM subscriptions
  QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY started_on, subscription_id, updated_at) = 1
)
SELECT COUNT(*) AS people,
       COUNT(*) FILTER (WHERE plan = 'plus_annual') AS annual,
       COUNT(*) FILTER (WHERE plan = 'plus_monthly' AND started_on > DATE '2026-05-31') AS june_starts,
       COUNT(*) FILTER (WHERE plan = 'plus_monthly' AND started_on <= DATE '2026-05-31') AS kept
FROM first_sub""").fetchone()
print(counts)
print(len(take_home), int(take_home["canceled"].sum()), round(float(take_home["canceled"].mean()), 3))

### The leakage check, part 1

In [ ]:
# The leakage check, part 1. The split comes first: 25 percent of the people are held out, with the same share of cancelers in both parts. Then a quick screen on the training people only: how well does each number column predict the outcome alone? A flag, not a verdict.
take_home["has_reason"] = take_home["cancel_reason"].notna().astype(int)           # 1 when a reason is filled in
take_home["annual_now"] = (take_home["plan_now"] == "plus_annual").astype(int)     # 1 when the latest saved row says annual
train, test = train_test_split(take_home, test_size=0.25, random_state=SEED, stratify=take_home["canceled"])
print(len(train), len(test), int(train["canceled"].sum()), int(test["canceled"].sum()))
CHECKED = ["days_to_buy", "activity_before", "visits_14d", "expenses_14d", "plus_uses_14d", "active_events", "plus_uses", "annual_now", "has_reason"]


def alone(col):
    """ROC AUC of one column used as a score on the training people, in either direction: 0.5 knows nothing, 1.0 knows everything."""
    a = roc_auc_score(train["canceled"], train[col])
    return round(float(max(a, 1 - a)), 3)


for col, score in sorted(((col, alone(col)) for col in CHECKED), key=lambda item: -item[1]):
    print(f"{col}: {score}")
print(train.groupby("annual_now")["canceled"].agg(["size", "sum"]).to_string())

### The leakage check, part 2

In [ ]:
# The leakage check, part 2: the held-out people score the model with and without each late column.
CATEGORIES = ["channel", "platform", "region"]
HONEST = ["days_to_buy", "activity_before", "visits_14d", "expenses_14d", "plus_uses_14d"]       # every one known by day 14


def held_out_auc(num):
    """Fit on the training people with these number columns and score the held-out people."""
    model = recipe(CATEGORIES, num).fit(train[CATEGORIES + num], train["canceled"])
    return round(float(roc_auc_score(test["canceled"], model.predict_proba(test[CATEGORIES + num])[:, 1])), 3)


print("honest columns:", held_out_auc(HONEST))
print("plus_uses added:", held_out_auc(HONEST + ["plus_uses"]))
print("active_events added:", held_out_auc(HONEST + ["active_events"]))
print("every column:", held_out_auc(HONEST + ["plus_uses", "active_events", "has_reason", "annual_now"]))

### Three candidates, then the held-out people once

In [ ]:
# Three candidates, compared by five-fold cross-validation on the training people only. Then the held-out people, used once.
CANDIDATES = {"channel only": (["channel"], []),
              "channel + 3 counts": (["channel"], ["visits_14d", "expenses_14d", "plus_uses_14d"]),
              "all honest columns": (CATEGORIES, HONEST)}
folds = StratifiedKFold(5, shuffle=True, random_state=SEED)
cv_means = {}
for name, (cat, num) in CANDIDATES.items():
    s = cross_val_score(recipe(cat, num), train[cat + num], train["canceled"], cv=folds, scoring="roc_auc")
    cv_means[name] = float(s.mean())
    print(f"{name}: {s.mean():.3f} ({s.min():.3f} to {s.max():.3f})")
chosen = max(cv_means, key=cv_means.get)
print("chosen:", chosen)

yt = test["canceled"].to_numpy()
cat, num = CANDIDATES[chosen]
p_model = recipe(cat, num).fit(train[cat + num], train["canceled"]).predict_proba(test[cat + num])[:, 1]
p_rule = recipe(["channel"], []).fit(train[["channel"]], train["canceled"]).predict_proba(test[["channel"]])[:, 1]       # the one-column rule: each channel's own cancel rate
p_flat = np.full(len(test), train["canceled"].mean())                                                                # the flat guess: everyone gets the training share
print([round(float(roc_auc_score(yt, p)), 3) for p in (p_flat, p_rule, p_model)])
print(auc_gain(p_model, p_rule, yt), auc_gain(p_model, p_flat, yt))
top = np.argsort(-p_model)[: len(test) // 5]                                                                         # the fifth of people with the highest chance
print(len(top), int(yt[top].sum()), round(float(yt[top].mean()), 3), round(float(yt.mean()), 3))

### How much the score depends on the split

In [ ]:
# How much does the held-out score depend on the split? The same model on twenty other splits (seeds 100 to 119): its AUC, the share of cancelers in its top fifth, its gain over the channel rule, and the held-out people of each channel.
others, fifths, gains = [], [], []
for seed in range(100, 120):
    a, b = train_test_split(take_home, test_size=0.25, random_state=seed, stratify=take_home["canceled"])
    m = recipe(cat, num).fit(a[cat + num], a["canceled"])
    p = m.predict_proba(b[cat + num])[:, 1]
    r = recipe(["channel"], []).fit(a[["channel"]], a["canceled"]).predict_proba(b[["channel"]])[:, 1]       # the one-column rule of the same split
    others.append(float(roc_auc_score(b["canceled"], p)))
    gains.append(others[-1] - float(roc_auc_score(b["canceled"], r)))
    fifths.append(float(b["canceled"].to_numpy()[np.argsort(-p)[: len(b) // 5]].mean()))
print(round(float(np.median(others)), 3), round(min(others), 3), round(max(others), 3), sum(o < roc_auc_score(yt, p_model) for o in others))
print(round(float(np.median(fifths)), 3), round(min(fifths), 3), round(max(fifths), 3))
print(round(float(np.median(gains)), 3), round(min(gains), 3), round(max(gains), 3), sum(g > 0 for g in gains))
print(test.groupby("channel")["canceled"].agg(["size", "sum"]).to_string())

## Run all from the top

Restart the session, run every cell from the first, then compare the numbers of the reference answer with the ones in your defense.